# Chunking and embeddings playground

Read a workbook, cut it into chunks, embed each chunk with Titan Text Embeddings v2, then retrieve the closest chunks
for a query. Every step is deliberately simple so it is easy to swap out.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").is_dir():  # walk up to the repo root wherever the kernel started
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")  # AWS_PROFILE locally; leave it unset where the machine has its own role

WORKBOOK = ROOT / "data" / "document_library" / "documents" / "raw" / "Location of 386 Pokémon in Gen 3.xlsx"
SHEET = "Gen 3 Location Sheet"

## 1. Read the sheet

In [ ]:
sheets = pd.read_excel(WORKBOOK, sheet_name=None, dtype=str, keep_default_na=False)
print(list(sheets))
df = sheets[SHEET]
df.head()

## 2. Chunk

Starting point: one chunk per row, written as `column: value` lines so each chunk carries its own headers. Empty cells are skipped.

In [ ]:
def row_to_chunk(row):
    return "\n".join(f"{col}: {val}" for col, val in row.items() if str(val).strip())

chunks = [row_to_chunk(row) for _, row in df.iterrows()]
chunks = [c for c in chunks if c]
print(f"{len(chunks)} chunks, longest {max(map(len, chunks)):,} characters\n")
print(chunks[0])

## 3. Embed

Same call as `amazon.titan-embed-text-v2:0` on the VDI. Titan v2 accepts up to 8,192 tokens per input; `normalize` makes cosine similarity a plain dot product.

In [ ]:
bedrock = boto3.Session().client("bedrock-runtime", region_name="us-east-1")

def embed(text):
    response = bedrock.invoke_model(modelId="amazon.titan-embed-text-v2:0",
                                    body=json.dumps({"inputText": text, "dimensions": 1024, "normalize": True}))
    return json.loads(response["body"].read())["embedding"]

with ThreadPoolExecutor(max_workers=8) as pool:
    vectors = np.array(list(pool.map(embed, chunks)))
vectors.shape

## 4. Retrieve

In [ ]:
def retrieve(query, k=5):
    scores = vectors @ np.array(embed(query))
    best = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in best]

QUERY = "Where can I find Ralts?"  # change me
for score, chunk in retrieve(QUERY):
    print(f"--- {score:.3f}\n{chunk[:500]}\n")

## 5. Your experiments